# Skew de Datos en PySpark

## ¿Qué es?

El *data skew* ocurre cuando los datos no se distribuyen uniformemente entre las particiones de un RDD/DataFrame. Algunas particiones (y por lo tanto algunas tareas) terminan con muchísimos más registros que otras, lo que puede ocasionar lo llamado *tareas rezagadas* (stragglers), alargan el job completo aunque el clúster tenga recursos libres.

**Causas comunes:**

- Joins sobre claves con distribución muy desigual (ejemplo: En la columna ID aparece en el 40% de las filas).

- **groupBy**/**reduceByKey** sobre columnas con pocos valores y muy concetrados.

- Particionamiento por hash de claves naturalmente sesgadas (fechas, países, IDs nulos o agrupados en la misma partición)

- Nulos: en muchos casos los valores **NULL** se concentran en una sola partición.

In [1]:
# Llamado de modulos

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
# Crear sesión de spark

spark = (
    SparkSession
    .builder
    .appName("Skew_Cache")
    .getOrCreate()
)

In [5]:
# Lectura de datos
Flight_csv = spark.read.csv(
    "/content/drive/MyDrive/flight data.csv",
    header = True,
    inferSchema = True
)


In [6]:
# Número de particiones

Flight_csv.rdd.getNumPartitions()

2

In [7]:
# Ver tamaño de partición

Flight_csv.rdd.glom().map(len).collect()

[499342, 499524]

In [8]:
# Verificar si es el total de observaciones

499342 + 499524

998866

In [9]:
# Tamaño ideal para 2 particiones

998866/2

499433.0

In [10]:
Flight_csv.count()

998866

## Estrategías de mitigación

In [11]:
# Configuraciones de mitigación para Skew Data
spark.conf.set("spark.sql.adaptive.enable", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewdPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewdPartitionThresholdInBytes", "256mb")

In [12]:
## Lectura de datos
Flight_csv = spark.read.csv(
    "/content/drive/MyDrive/flight data.csv",
    header = True,
    inferSchema = True
)

In [13]:
# Número de particiones

Flight_csv.rdd.getNumPartitions()

2

In [14]:
# Ver tamaño de partición

Flight_csv.rdd.glom().map(len).collect()

[499342, 499524]

# Ejercicio 1:

En los datos de vuelos (flight), hay hubs grandes que concentran muchísimas más filas que otros aeropuertos. Es el caso para un skew de datos ¿Esto será cierto?

In [15]:
# Ver distribución de la clave candidata a particionar

Flight_csv.groupBy("from_airport_code")\
          .count()\
          .orderBy(desc("count"))\
          .show(20)

+-----------------+-----+
|from_airport_code|count|
+-----------------+-----+
|              YYZ|55056|
|              MUC|54828|
|              FRA|53992|
|              CPH|53745|
|              CDG|52340|
|              BRU|48776|
|              DUB|48634|
|              VIE|48284|
|              ATH|48084|
|              DEL|47567|
|              SYD|47091|
|              MEL|44636|
|              BOG|43324|
|              CAI|42330|
|              SCL|37025|
|              GRU|33280|
|              CNF|30220|
|              PVG|25667|
|              AEP|21483|
|              ADD|19572|
+-----------------+-----+
only showing top 20 rows


In [16]:
# Ver tamaño real de partición tras una repartición mediante esa columna

Flight_airpot_code = Flight_csv.repartition("from_airport_code")
Tamaños = Flight_airpot_code.rdd.glom().map(len).collect()
print(sorted(Tamaños, reverse=True)[:10])

[503730, 475564, 19572]


In [21]:
# Forzar y resolver skew en un groupBy
spark.conf.set("spark.sql.adaptive.enable", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

Codigo = (
    Flight_csv
    .groupBy("from_airport_code")
    .agg(
        count("*").alias("n_vuelos"),
        avg("price").alias("Precio_promedio"),
        avg("co2_emissions").alias("CO2_promedio")
    )
)

Codigo.explain(True)


== Parsed Logical Plan ==
'Aggregate ['from_airport_code], ['from_airport_code, 'count(*) AS n_vuelos#241, 'avg('price) AS Precio_promedio#242, 'avg('co2_emissions) AS CO2_promedio#243]
+- Relation [from_airport_code#92,from_country#93,dest_airport_code#94,dest_country#95,aircraft_type#96,airline_number#97,airline_name#98,flight_number#99,departure_time#100,arrival_time#101,duration#102,stops#103,price#104,currency#105,co2_emissions#106,avg_co2_emission_for_this_route#107,co2_percentage#108,scan_date#109] csv

== Analyzed Logical Plan ==
from_airport_code: string, n_vuelos: bigint, Precio_promedio: double, CO2_promedio: double
Aggregate [from_airport_code#92], [from_airport_code#92, count(1) AS n_vuelos#241L, avg(price#104) AS Precio_promedio#242, avg(co2_emissions#106) AS CO2_promedio#243]
+- Relation [from_airport_code#92,from_country#93,dest_airport_code#94,dest_country#95,aircraft_type#96,airline_number#97,airline_name#98,flight_number#99,departure_time#100,arrival_time#101,duratio

In [22]:
Codigo.rdd.getNumPartitions()

1

In [24]:
Codigo.show(100)

+-----------------+--------+------------------+------------------+
|from_airport_code|n_vuelos|   Precio_promedio|      CO2_promedio|
+-----------------+--------+------------------+------------------+
|              ALG|   15737|1528.1926669632078| 944409.0358758682|
|              VIE|   48284|1072.4092038770607| 774845.7897359242|
|              GRU|   33280|2468.6396334134615|1599105.8169065183|
|              CTU|   14334| 3328.786870378122|1535774.7144267382|
|              PEK|    8071|3202.9524104597845|1420239.3332504043|
|              SHA|   16527|3466.6211229618734|1421279.2820200792|
|              SYD|   47091| 2100.679429190291|1744971.5481886535|
|              PVG|   25667| 3225.718579983561| 1431356.539093684|
|              BRU|   48776| 998.1845784812202| 765441.4840074775|
|              YYZ|   55056|1266.4049149956409| 945948.9446870452|
|              CNF|   30220|2162.2189278623428|1439501.8593532108|
|              VCP|   12479| 3495.681705264845|1697648.3410803

# Cache y Persist en PySpark

Spark evalúa todo de forma **perezoza** (lazy): las trasnformaciones (select, filter, join, etc) solo constituyen un plan lógico (lineage). Solo al ejecutar una acción (count, collect, write, etc) se materializa el cálculo. Si reutilizas el mismo DataFrame en varias acciones, Spark **recalcular** todo el lineage cada que vez que se procese, ahí entra el **Cache** / **persist**

## ¿Cuando usar cache/pesist?

1) Cuando un DataFrame se reutiliza en múltiples acciones o ramas del pipeline (ej. Para entrenar varios modelos, o calcular varias métricas sobre el mismo resultado intermedio).

2) Después de una operación costosa (ej. un join grande, una agregación) que alimenta varios pasos anteriores.

3) En loops iterativos (ML, grafos) donde el mismo DataFrame se referencia en cada iteración.

## ¿Cuando no usarlo?

- Si el dataframe se usa una sola vez (solo se agrega el overhead).

- Si los datos no caben ni en la memoria (RAM) ni en el disco de forma razonable.

- Justo antes de un write final sin reutilización posterior.

In [29]:
# Ejercicio de persist

## Procesamiento con flight

Flight_val = (
    Flight_csv
    .filter(col("stops") == 0)
    .filter(col("co2_percentage") != "0%")
)

Flight_val.persist(StorageLevel.MEMORY_AND_DISK)
Flight_val.count() # Materializa el cache (acción "barata de calentamiento")

11499

In [30]:
# Reporte 1: Precio promedio por país de destino

reporte_precio = (
    Flight_val
    .groupBy("dest_country")
    .agg(avg("price").alias("Precio_promedio"))
)

reporte_precio.show()

+-------------+------------------+
| dest_country|   Precio_promedio|
+-------------+------------------+
|       Sweden|329.35377358490564|
|  Philippines|          800.9375|
|     Malaysia|478.32142857142856|
|    Singapore| 963.8764705882353|
|       Turkey| 341.2183622828784|
|      Germany|343.10450160771705|
|       France| 437.9069212410501|
|       Greece|229.70935960591132|
|       Taiwan| 555.5714285714286|
|       Dublin| 219.0473372781065|
|    Argentina| 292.2238805970149|
|      Belgium|302.49333333333334|
|        Qatar| 583.1242603550296|
|         Peru|            401.35|
|        India| 162.5910478128179|
|United States| 972.6911290322581|
|        China| 363.4604166666667|
|        Chile|422.27027027027026|
|        Italy|193.65196078431373|
|       Norway|253.84924623115577|
+-------------+------------------+
only showing top 20 rows


In [31]:
# Reporte 2: Escritura final

(Flight_val
 .write
 .mode("overwrite")
 .partitionBy("from_country")
 .parquet("salida/vuelos_validos"))

Flight_val.unpersist()

DataFrame[from_airport_code: string, from_country: string, dest_airport_code: string, dest_country: string, aircraft_type: string, airline_number: string, airline_name: string, flight_number: string, departure_time: timestamp, arrival_time: timestamp, duration: int, stops: int, price: double, currency: string, co2_emissions: int, avg_co2_emission_for_this_route: int, co2_percentage: string, scan_date: timestamp]

In [32]:
# Ejercicio de cache

## Procesamiento con flight

Flight_val = (
    Flight_csv
    .filter(col("stops") == 0)
    .filter(col("co2_percentage") != "0%")
)

Flight_val.cache()
Flight_val.count() # Materializa el cache

11499

In [33]:
# Reporte 1: Precio promedio por país de destino

reporte_precio = (
    Flight_val
    .groupBy("dest_country")
    .agg(avg("price").alias("Precio_promedio"))
)

reporte_precio.show()

+-------------+------------------+
| dest_country|   Precio_promedio|
+-------------+------------------+
|       Sweden|329.35377358490564|
|  Philippines|          800.9375|
|     Malaysia|478.32142857142856|
|    Singapore| 963.8764705882353|
|       Turkey| 341.2183622828784|
|      Germany|343.10450160771705|
|       France| 437.9069212410501|
|       Greece|229.70935960591132|
|       Taiwan| 555.5714285714286|
|       Dublin| 219.0473372781065|
|    Argentina| 292.2238805970149|
|      Belgium|302.49333333333334|
|        Qatar| 583.1242603550296|
|         Peru|            401.35|
|        India| 162.5910478128179|
|United States| 972.6911290322581|
|        China| 363.4604166666667|
|        Chile|422.27027027027026|
|        Italy|193.65196078431373|
|       Norway|253.84924623115577|
+-------------+------------------+
only showing top 20 rows


In [34]:
# Reporte 2: Escritura final

(Flight_val
 .write
 .mode("overwrite")
 .partitionBy("from_country")
 .parquet("salida2/vuelos_validos"))

Flight_val.unpersist()

DataFrame[from_airport_code: string, from_country: string, dest_airport_code: string, dest_country: string, aircraft_type: string, airline_number: string, airline_name: string, flight_number: string, departure_time: timestamp, arrival_time: timestamp, duration: int, stops: int, price: double, currency: string, co2_emissions: int, avg_co2_emission_for_this_route: int, co2_percentage: string, scan_date: timestamp]

In [35]:
# Fin

spark.stop()